In [ ]:
!pip -q install scikit-survival pandas numpy scipy scikit-learn
print('deps ok')

In [ ]:
import os, sys, glob, zipfile
from google.colab import files
up = files.upload()                                  # choose lg-piml-github.zip
zname = [k for k in up if k.lower().endswith('.zip')][0]
zipfile.ZipFile(zname).extractall('/content/code')
cfg = glob.glob('/content/code/**/config.py', recursive=True)
assert cfg, 'config.py not found in the zip — did you upload lg-piml-github.zip?'
CODEDIR = os.path.dirname(cfg[0]); os.chdir(CODEDIR); sys.path.insert(0, CODEDIR)
print('code dir:', CODEDIR)

In [ ]:
import urllib.request, os
os.makedirs('CMAPSSData', exist_ok=True)
url = 'https://raw.githubusercontent.com/ericlrf/rul/main/CMAPSSData/train_FD003.txt'
dst = 'CMAPSSData/train_FD003.txt'
if not os.path.exists(dst) or os.path.getsize(dst) < 1000:
    urllib.request.urlretrieve(url, dst)
print(dst, os.path.getsize(dst), 'bytes  ->  OK')

In [ ]:
import os, sys, importlib
os.environ['LG_OUT'] = '/content/results'; os.makedirs('/content/results', exist_ok=True)
for m in ['config','engines','cmapss_data','cmapss_fd_run','bootstrap_ci']:
    sys.modules.pop(m, None)
import cmapss_fd_run, bootstrap_ci
cmapss_fd_run.RES = os.path.join(os.environ['LG_OUT'], 'results_cmapss_fd.json')  # belt & suspenders
os.environ.pop('CMAPSS_FAST', None)              # ensure FULL training
print('ready — LG_OUT =', os.environ['LG_OUT'])

In [ ]:
cmapss_fd_run.prep('FD003')
bootstrap_ci.fit_preds('FD003', budget=10**9, nfast=0)   # 7 full fits (the slow part)
table, boot = bootstrap_ci.full_report('FD003', n_boot=2000)

In [ ]:
from google.colab import files
files.download('/content/results/results_full_FD003.json')